# 📘 Notebook 09 · 机器学习与深度学习挖掘 Alpha

> 这是头部量化私募**现在主要在做的事**。LightGBM 是基础线，2020 年起的真正前沿是：**Transformer + GNN + 自监督**。

**学完你会：**

- 知道传统 ML vs 深度学习在金融上的真实差距
- 用 LSTM 做时序股价预测
- 用 Transformer (TFT 风格) 做多资产联合建模
- 用 GNN 建模股票之间的关系（行业图谱、供应链、关联交易）
- 用 AutoEncoder 自动挖掘因子
- 知道为什么"AI 找因子"在 80% 公司只是营销词

**预计 10-15 小时。**

**前置**：Notebook 01 + 08 + 你已经掌握 PyTorch 基础（如果不会，先看 Karpathy 的 [makemore](https://youtu.be/PaCmpygFfXo)）。

## 1. 现状：头部私募在用什么

### 1.1 公开信息能猜到的

| 公司 | 已知方向 |
|---|---|
| 幻方 | 深度学习 alpha，自研框架 + 自建千卡集群 |
| 明汯 | 全因子模型 + 高频做市 |
| 九坤 | 多策略 + ML 选股 |
| 灵均 | 高频统计套利 + 多因子 |
| Renaissance | 据传用很多隐马尔可夫 + 信号分解 |
| Two Sigma | 全员博士+ML，另类数据+深度模型 |
| Citadel | Multi-PM 模式，不同小组用不同方法 |

### 1.2 ML/DL 在量化里的"五代"

```
G1 (2010-)  : LASSO / Ridge / OLS + 因子合成
G2 (2015-)  : XGBoost / LightGBM 截面选股
G3 (2018-)  : LSTM / GRU 时序建模
G4 (2020-)  : Transformer / GNN 多资产 + 关系建模
G5 (2023-)  : 大模型 + 自监督预训练（量化里的 GPT 时代）
```

**重要事实**：G5 在公开 paper 中表现普遍**不如 G2 + 好的特征工程**。原因是金融数据信噪比太低，模型 capacity 越大越容易过拟合。

**但头部私募仍然在投入 G5**，因为：
- 在自家几十亿条特征数据上，capacity 才能体现
- 他们有研发资源烧
- 即使 marginal 提升也值得（管理费 + 业绩报酬）

### 1.3 给学习者的建议

**对应届生**：
- 必会 G2（LightGBM）
- 应会 G3（LSTM 简单架构）
- 加分项 G4（Transformer / GNN 各做过一个项目）
- G5 不需要会，知道存在就行

下面我们从 G3 开始过一遍。

## 2. LSTM 时序建模

### 2.1 为什么不直接 MLP

价量数据是**有顺序**的。MLP 把过去 20 天展平成 20 个特征，丢失了顺序结构。

LSTM 通过门控机制保留长程依赖：

$$
\begin{aligned}
f_t &= \sigma(W_f \cdot [h_{t-1}, x_t] + b_f) \quad \text{(遗忘门)} \\
i_t &= \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) \quad \text{(输入门)} \\
\tilde{C}_t &= \tanh(W_C \cdot [h_{t-1}, x_t] + b_C) \\
C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \\
o_t &= \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) \\
h_t &= o_t \odot \tanh(C_t)
\end{aligned}
$$

不需要背公式，关键直觉：**门 = 决定多少历史信息保留**。

### 2.2 LSTM 选股的标准管线

```
输入   : (B, T, F)  B个样本，T天，F个特征
LSTM   : (B, T, H)  H=hidden_size
取最后 : (B, H)
全连接 : (B, 1)     预测下期收益排序分数
```

In [ ]:
# 检查 PyTorch
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    HAS_TORCH = True
    print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
except ImportError:
    HAS_TORCH = False
    print('需要 PyTorch: pip install torch')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ---------- 构造数据：300 只股票 × 1000 天 × 8 个特征 ----------
np.random.seed(42)
n_stocks, n_days, n_feat = 300, 1000, 8

# 让其中 4 个特征对未来收益真的有信号
factor_load = np.random.randn(n_stocks, 4) * 0.3
real_signal = np.zeros((n_days, n_stocks))
for t in range(20, n_days):
    feats_lag = np.random.randn(4, n_stocks) * 0.5  # 滞后特征
    real_signal[t] = (factor_load @ feats_lag).diagonal() if factor_load.shape[1] == 4 else 0

# 模拟 8 个特征 + 1 个目标（未来 5 日收益）
X_raw = np.random.randn(n_days, n_stocks, n_feat).astype(np.float32)
fwd_ret = (real_signal * 0.001 + np.random.randn(n_days, n_stocks) * 0.02).astype(np.float32)

print(f'X 形状: {X_raw.shape}')
print(f'y 形状: {fwd_ret.shape}')

In [ ]:
if HAS_TORCH:
    # ---------- 把数据切成 LSTM 输入: 每个样本是 (T_window, F) ----------
    T_window = 20

    def make_samples(X_raw, y, T):
        samples_X, samples_y = [], []
        for t in range(T, X_raw.shape[0]):
            for s in range(X_raw.shape[1]):
                samples_X.append(X_raw[t-T:t, s, :])
                samples_y.append(y[t, s])
        return np.array(samples_X), np.array(samples_y)

    # 训练集前 600 天，测试集后 380 天
    Xs_tr, ys_tr = make_samples(X_raw[:600], fwd_ret[:600], T_window)
    Xs_te, ys_te = make_samples(X_raw[600:], fwd_ret[600:], T_window)
    print(f'训练样本: {Xs_tr.shape} → {ys_tr.shape}')
    print(f'测试样本: {Xs_te.shape} → {ys_te.shape}')

    # ---------- 模型 ----------
    class LSTMAlpha(nn.Module):
        def __init__(self, n_feat=8, hidden=32, n_layers=2):
            super().__init__()
            self.lstm = nn.LSTM(n_feat, hidden, n_layers, batch_first=True, dropout=0.2)
            self.head = nn.Sequential(
                nn.Linear(hidden, 16), nn.GELU(),
                nn.Linear(16, 1)
            )
        def forward(self, x):
            out, _ = self.lstm(x)
            return self.head(out[:, -1, :]).squeeze(-1)   # 取最后时刻

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = LSTMAlpha(n_feat=n_feat).to(device)
    print(f'\n模型参数量: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
if HAS_TORCH:
    # ---------- 训练 ----------
    opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10)
    loss_fn = nn.MSELoss()

    X_tr_t = torch.from_numpy(Xs_tr).to(device)
    y_tr_t = torch.from_numpy(ys_tr).to(device)
    X_te_t = torch.from_numpy(Xs_te).to(device)
    y_te_t = torch.from_numpy(ys_te).to(device)

    # 简化：用 mini-batch
    batch = 512
    losses = []
    for epoch in range(10):
        model.train()
        perm = torch.randperm(len(X_tr_t))
        running = 0
        for i in range(0, len(X_tr_t), batch):
            idx = perm[i:i+batch]
            x, y = X_tr_t[idx], y_tr_t[idx]
            opt.zero_grad()
            pred = model(x)
            loss = loss_fn(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            running += loss.item() * len(idx)
        scheduler.step()
        train_loss = running / len(X_tr_t)

        # 测试
        model.eval()
        with torch.no_grad():
            te_pred = model(X_te_t)
            te_loss = loss_fn(te_pred, y_te_t).item()
            # 横截面 IC（粗算：把测试集 reshape 成 (T, stocks)）
            pred_np = te_pred.cpu().numpy()
            n_test_days = len(pred_np) // n_stocks
            pred_mat = pred_np[:n_test_days * n_stocks].reshape(n_test_days, n_stocks)
            actual_mat = ys_te[:n_test_days * n_stocks].reshape(n_test_days, n_stocks)
            ic_per_day = [np.corrcoef(pred_mat[t], actual_mat[t])[0,1] for t in range(n_test_days)]
            ic = np.nanmean(ic_per_day)
        losses.append((train_loss, te_loss, ic))
        print(f'Epoch {epoch+1:2d}  train_loss={train_loss:.6f}  test_loss={te_loss:.6f}  IC={ic:.4f}')

**模拟数据上 IC 不会很惊艳**（我们生成时只埋了弱信号）。**真实数据上 LSTM 比 LGBM 一般好 0.005-0.02 IC**。

这看起来微小，但在 4000 只股票 × 250 天上**乘出来就是真金白银**。

### 2.3 LSTM 实战的几个坑

1. **不要做单只股票的时序预测**——单股信噪比太低
2. **要做"截面"任务**：每天对所有股票预测一个相对得分，做 cross-sectional IC
3. **特征 normalize 要谨慎**：用 expanding window 而不是 train set 全局，避免 lookahead
4. **早停 + 学习率退火**：金融数据训 5-10 epoch 就够，过多必过拟合
5. **多模型集成**：单个 LSTM 不稳定，5-10 个 random seed 平均

## 3. Transformer：现代标配

### 3.1 Self-Attention 公式

$$\text{Attn}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

- $Q, K, V$ 都是输入的线性变换
- $QK^T$ 给出 token 间的"相关度"
- softmax 归一化 → 加权聚合 $V$

**比 LSTM 强在哪：**
- **并行计算**（LSTM 必须按时间逐步）
- **全局注意力**（任意两个时刻可直接交互，不像 LSTM 信息要逐层传递）
- **可解释**（attention map 可视化）

### 3.2 量化 Transformer 的两个典型架构

**A. 单股票时序 Transformer**（小改 NLP Encoder）

```
Input: (B, T, F)  
  ↓ Linear + Positional Encoding
  ↓ N × TransformerEncoderLayer
  ↓ Take last token
  ↓ Output: (B, 1)
```

**B. 横截面 Transformer**（CN Transformer）

每天把所有股票看成一个序列，让股票之间互相注意：

```
Input: (B, N_stocks, F)  
  ↓ Self-Attention (在 stock 维度)
  ↓ 每只股票得到一个考虑了其他股票信息的表示
  ↓ Output: (B, N_stocks, 1)
```

**横截面 Transformer 是 2022 后的新趋势**——能同时建模"时序 + 截面"。代表 paper: HIST, DLinear, FactorVAE。

### 3.3 一个简化的 Transformer 时序模型

In [ ]:
if HAS_TORCH:
    class TransformerAlpha(nn.Module):
        def __init__(self, n_feat=8, d_model=64, n_heads=4, n_layers=2, T=20):
            super().__init__()
            self.input_proj = nn.Linear(n_feat, d_model)
            self.pos_embed = nn.Parameter(torch.randn(1, T, d_model) * 0.02)
            layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=n_heads,
                dim_feedforward=d_model*4, dropout=0.1,
                batch_first=True, norm_first=True,    # Pre-LN，训练更稳
            )
            self.encoder = nn.TransformerEncoder(layer, n_layers)
            self.head = nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, 16), nn.GELU(),
                nn.Linear(16, 1)
            )

        def forward(self, x):
            h = self.input_proj(x) + self.pos_embed[:, :x.size(1)]
            h = self.encoder(h)
            return self.head(h[:, -1, :]).squeeze(-1)

    model2 = TransformerAlpha().to(device)
    print(f'Transformer 参数量: {sum(p.numel() for p in model2.parameters()):,}')

    # 快速跑 3 epoch 看趋势
    opt2 = optim.AdamW(model2.parameters(), lr=1e-3, weight_decay=1e-5)
    for epoch in range(3):
        model2.train()
        perm = torch.randperm(len(X_tr_t))
        running = 0
        for i in range(0, len(X_tr_t), batch):
            idx = perm[i:i+batch]
            opt2.zero_grad()
            pred = model2(X_tr_t[idx])
            loss = loss_fn(pred, y_tr_t[idx])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model2.parameters(), 1.0)
            opt2.step()
            running += loss.item() * len(idx)
        print(f'Transformer Epoch {epoch+1}  train_loss={running/len(X_tr_t):.6f}')

### 3.4 真实使用的 Transformer 变体

| 模型 | 论文 | 用途 |
|---|---|---|
| **Informer** | AAAI 2021 | 长序列预测 |
| **Autoformer** | NeurIPS 2021 | 分解时序为趋势 + 季节 |
| **PatchTST** | ICLR 2023 | NLP-style patch，性能强 |
| **iTransformer** | ICLR 2024 | 把"时间"和"变量"维度倒过来 attention |
| **TimesNet** | ICLR 2023 | 时序看成 2D 图像 |
| **DLinear** | AAAI 2023 | 一个 trivial 的线性模型打败很多 Transformer！|

**特别提 DLinear**：Zeng et al. 2023 证明在很多时序预测 benchmark 上，**一个简单的线性模型** ($\hat y_{t+1:t+h} = W \cdot y_{t-L:t}$) 比所有花哨的 Transformer 都好。

→ **这印证了金融数据的特点：信号太弱、太短，简单模型 generalization 更好。**

## 4. 图神经网络 (GNN)：建模股票关系

### 4.1 为什么需要 GNN

传统模型把股票当独立样本。但实际中：
- 同行业公司高度相关（茅台 / 五粮液）
- 上下游供应链（苹果 / 立讯精密）
- 持股关联（华兴源创 / 中芯国际）
- 主题概念（AI / 大模型相关板块）

**这些关系是图结构**：节点=股票，边=关系。GNN 利用图结构 message passing 建模。

### 4.2 GNN 基础：GCN

每一层更新节点表示：

$$h_v^{(l+1)} = \sigma\left(\sum_{u \in \mathcal{N}(v) \cup \{v\}} \frac{1}{\sqrt{|N(v)||N(u)|}} W^{(l)} h_u^{(l)}\right)$$

**直觉**：每个节点的新表示 = 邻居表示的加权平均，经过线性变换 + 激活。

### 4.3 量化中常用的图

| 图 | 边的定义 |
|---|---|
| 行业图 | 同 GICS 行业 |
| 关联图 | 关联交易、控股关系 |
| 供应链图 | 上下游订单数据 |
| 持仓相关 | 基金共同持有 |
| 相关性图 | 历史收益相关性 > 阈值 |
| 主题图 | 财报 / 新闻共现 |

### 4.4 实战提醒

GNN 在量化里**没那么神**：

- 真实股票图边数 $\sim O(N \log N)$，N=4000 时图很稀疏
- 行业关系 LightGBM 加个 industry-dummy 已经捕捉到 80%
- GNN 真正发力是**长尾关系**（如新出现的概念股 + 老股票的关联）
- 大部分头部私募的 "GNN factor" 在标准基准上比 LGBM 提升 < 0.005 IC

In [ ]:
if HAS_TORCH:
    # 一个最简 GCN 层
    class GCNLayer(nn.Module):
        def __init__(self, in_dim, out_dim):
            super().__init__()
            self.linear = nn.Linear(in_dim, out_dim)
        def forward(self, h, A_hat):
            # A_hat: 归一化邻接矩阵 (N, N)
            # h: 节点特征 (N, F)
            return self.linear(A_hat @ h)

    class GCNAlpha(nn.Module):
        def __init__(self, n_feat=8, hidden=32):
            super().__init__()
            self.g1 = GCNLayer(n_feat, hidden)
            self.g2 = GCNLayer(hidden, hidden)
            self.head = nn.Linear(hidden, 1)
        def forward(self, x, A_hat):
            h = torch.relu(self.g1(x, A_hat))
            h = torch.relu(self.g2(h, A_hat))
            return self.head(h).squeeze(-1)

    # 模拟一个图
    N = 100
    A = (np.random.rand(N, N) > 0.95).astype(np.float32)
    np.fill_diagonal(A, 1)
    A = (A + A.T > 0).astype(np.float32)   # 无向化
    # GCN 归一化: D^(-1/2) A D^(-1/2)
    D = A.sum(axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(D + 1e-6))
    A_hat = D_inv_sqrt @ A @ D_inv_sqrt

    x = torch.randn(N, 8)
    A_hat_t = torch.from_numpy(A_hat)
    gcn = GCNAlpha()
    out = gcn(x, A_hat_t)
    print(f'GCN 输出形状: {out.shape}  (每个节点一个预测分数)')
    print(f'前 5 个节点的预测: {out[:5].detach().numpy()}')

## 5. AutoEncoder：自动挖掘因子

### 5.1 思想

传统因子是**人工设计**。AutoEncoder 是：

```
原始数据 (1000 个特征) 
  ↓ Encoder
潜在表示 (10 维) ← 这就是"自动挖到的因子"
  ↓ Decoder
重构数据
```

如果潜在表示能"还原"原始数据，说明它捕捉了核心信息。

### 5.2 在金融上的变种

| 变种 | 思想 |
|---|---|
| **Vanilla AE** | 重构损失最小化 |
| **VAE** | 加上 KL 散度正则，潜在变量为正态 |
| **Contrastive AE** | 让相似样本的潜在表示也相似 |
| **Beta-VAE** | 强制 disentanglement |
| **FactorVAE** (2021) | 专为金融的 VAE，潜在变量直接对应因子 |

### 5.3 何时用 AutoEncoder

✅ 数据极高维（如几千个原始特征）→ 降维
✅ 想找隐藏的潜在结构（如"行业风险因子"）
✅ 数据有时空相关性（卫星图像、订单簿）

❌ 你已经有几十个手工因子 → 直接用 LightGBM 更好
❌ 数据稀疏（如基本面）→ AE 不擅长

In [ ]:
if HAS_TORCH:
    class AutoFactor(nn.Module):
        """用 AutoEncoder 从高维特征自动提取因子"""
        def __init__(self, n_feat=50, latent=8):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Linear(n_feat, 32), nn.ReLU(),
                nn.Linear(32, 16), nn.ReLU(),
                nn.Linear(16, latent),
            )
            self.decoder = nn.Sequential(
                nn.Linear(latent, 16), nn.ReLU(),
                nn.Linear(16, 32), nn.ReLU(),
                nn.Linear(32, n_feat),
            )
        def forward(self, x):
            z = self.encoder(x)
            x_hat = self.decoder(z)
            return x_hat, z

    # 模拟高维特征
    n_samples = 5000
    n_feat = 50
    X = torch.randn(n_samples, n_feat)
    # 让特征有低维结构（隐藏 5 个因子）
    true_factors = torch.randn(n_samples, 5)
    true_loadings = torch.randn(5, n_feat)
    X = true_factors @ true_loadings + 0.3 * torch.randn(n_samples, n_feat)

    model3 = AutoFactor(n_feat=n_feat, latent=8)
    opt3 = optim.Adam(model3.parameters(), lr=1e-3)

    for epoch in range(50):
        opt3.zero_grad()
        x_hat, z = model3(X)
        loss = ((x_hat - X) ** 2).mean()
        loss.backward()
        opt3.step()
        if (epoch+1) % 10 == 0:
            print(f'Epoch {epoch+1}  recon_loss={loss.item():.4f}')

    # 训练完，z 就是"自动找到的 8 个因子"
    _, z = model3(X)
    z_np = z.detach().numpy()
    print(f'\n自动因子形状: {z_np.shape}')
    print(f'前 5 个样本的因子值:\n{z_np[:5]}')

    # 检查这 8 个因子能不能解释原始特征方差
    var_explained = 1 - ((model3(X)[0] - X)**2).mean().item() / X.var().item()
    print(f'\n8 个因子能解释 {var_explained*100:.1f}% 的方差（vs 原始 50 个特征）')

## 6. 自监督预训练：量化里的"GPT 时刻"

### 6.1 NLP 的启示

GPT 的成功：先在海量文本上**自监督预训练**（预测下一个 token），再在小数据上 fine-tune。

类比到金融：

```
预训练: 海量历史数据（无标签）   →  学到通用表示
  ↓
  Fine-tune: 小数据 + 具体任务（如 IC 最大化）
  ↓
  目标策略
```

### 6.2 金融自监督的几种任务

**1. Masked Reconstruction**

把价量序列中的某一段"挖空"，让模型预测被挖的部分。

**2. Next Patch Prediction**

把价量切成 patch，预测下一个 patch（类似 GPT）。

**3. Contrastive Learning (SimCLR style)**

- 同一只股票的两段时间作为正样本
- 不同股票作为负样本
- 让正样本表示靠近、负样本远离

**4. Reconstruction + Forecasting (TimeMAE)**

同时做"重构"和"预测"双任务。

### 6.3 公开 paper / 项目

| 项目 | 简介 |
|---|---|
| **Kronos**（2024）| 金融时序自监督预训练，Tsinghua 团队 |
| **FinBERT** | 金融文本 BERT |
| **StockGPT**（实验性）| GPT 风格的股票预测 |
| **Lag-Llama**（2023）| 时序基础模型 |

**说实话**：截止 2026 年，**没有任何公开的"金融基础模型"在严格 walk-forward 评估下显著打败 LightGBM**。但巨头都在投入，**未来 2-3 年可能突破**。

### 6.4 给学习者的策略

- 现在做：把 G2-G4 的几个 baseline 做熟
- 跟踪：每月扫 [arXiv q-fin](https://arxiv.org/list/q-fin/recent)
- 长期：如果你在头部私募，会有机会接触自监督的真实项目

## 7. "因子动物园"问题与解决

### 7.1 问题

学术界已发现 **400+ 因子**（Harvey 2016 综述），但其中**大部分在样本外失效**。

原因：
- p-hacking（试多了挑显著的）
- 已被市场套利消化
- 数据窥探

### 7.2 检测真假 Alpha 的最佳实践（López de Prado）

1. **Deflated Sharpe Ratio**：考虑你试了多少策略后的 Sharpe 显著性
2. **Probabilistic Sharpe Ratio**：t 检验的 Sharpe 版本
3. **Combinatorial Purged Cross-Validation (CPCV)**：比 walk-forward 更鲁棒
4. **Backtest Overfitting Probability (PBO)**：策略上线后崩盘的概率

### 7.3 防过拟合"七大守则"

```
1. 因子数量 < 样本数 / 50（严格）
2. 任何超参 grid search 之后，再做一次完全独立的样本外验证
3. 不要"试出来"模型架构——理论先行
4. 多个 random seed 平均（10+ 个）
5. 不同年份、不同市场都要验证
6. 真实交易成本 + 滑点放进回测
7. 上线前先模拟盘 3-6 个月
```

## 8. 头部私募"AI Alpha"的真相

> "我们用 AI 找因子" 这句话在 2024-2025 的招聘宣传里出现 90% 以上。**实情是什么？**

### 8.1 大部分公司在做的（约 80%）

- **LightGBM / XGBoost + 几百个手工特征**：仍然是 SOTA 基线
- **LSTM + MLP head**：上一代标配
- **少量 Transformer 实验**：通常是"小组项目"，不是主力

→ 名字叫 "AI"，本质 G2-G3。

### 8.2 真正在做前沿的（约 15%）

- 横截面 Transformer + 自监督预训练
- 多模态：价量 + 财报文本 + 卫星 + 公告
- 强化学习用于组合配置 + 执行

### 8.3 在烧钱做未来的（约 5%）

- 千卡集群预训练"金融基础模型"
- 在线学习 / 持续学习
- 因果发现 + ML 结合（你研究的 DID + ML 方向）

### 8.4 求职启示

**面试时被问"你用过什么 AI 方法"**：

- **不要堆名词**："我会 LSTM、Transformer、GNN、VAE、Diffusion..."→ 一眼假
- **讲一个深做的项目**："我做了 LSTM 对 A 股月频选股，最终改进了什么具体地方"
- **主动承认局限**：金融数据信噪比低，复杂模型 generalization 难

**展现真正的专业**：

- 区分 IC、Rank IC、IR
- 区分 walk-forward 和 purged CV
- 知道 DLinear paper 的"打脸"结果
- 能讲清楚自己的 ablation study

## 9. 一个完整的现代化深度因子流水线

下面是当代私募研究员**一个完整 ML 项目**的标准流程：

```
1. 数据接入
   ├─ 价量（分钟 / 日频）
   ├─ 基本面（PIT 财报）
   ├─ 分析师预期（Wind / 同花顺）
   ├─ 资金流（龙虎榜、北向）
   ├─ 文本（新闻、研报、公告）
   └─ 另类（卫星、信用卡、爬虫）
   
2. 特征工程
   ├─ 手工特征（动量、波动、量价相关）
   ├─ 自动特征（PCA / AutoEncoder / TSFresh）
   └─ 标签：未来 N 日 cross-sectional rank
   
3. 模型设计
   ├─ Baseline: LightGBM
   ├─ DL Baseline: LSTM + MLP
   ├─ 进阶: PatchTST / iTransformer / GNN
   └─ Ensemble: 多种子 + 多模型平均
   
4. 训练
   ├─ Walk-forward (rolling window)
   ├─ Purged k-fold (label overlap 处理)
   ├─ EMA weights (训练稳定)
   └─ Multi-task (主任务 + auxiliary)
   
5. 评估
   ├─ IC, Rank IC, IR
   ├─ Top/Bottom quintile spread
   ├─ Turnover (换手率)
   ├─ Capacity (容量分析)
   └─ Subsample stability (行业、市值、年份)
   
6. 上线
   ├─ 模拟盘 (paper trading) 3-6 月
   ├─ 实盘小资金 6-12 月
   ├─ 全量上线
   └─ Online learning / 定期重训
```

**你不需要每一步都做满**——挑 2-3 个深做，简历就有故事讲。

## 10. 推荐资源

**论文必读：**
- López de Prado《Advances in Financial Machine Learning》(2018)
- Gu, Kelly, Xiu (2020) "Empirical Asset Pricing via Machine Learning" RFS
- Kelly, Pruitt, Su (2019) "Characteristics Are Covariances" JFE
- 蚂蚁集团《Kronos: A Foundation Model for Chinese Financial Time Series》(2024)
- DLinear (Zeng et al. 2023) "Are Transformers Effective for Time Series Forecasting?"
- PatchTST (Nie et al. 2023)

**代码库：**
- [Qlib](https://github.com/microsoft/qlib)：微软开源量化平台，自带数据 + 多种模型
- [FinRL](https://github.com/AI4Finance-Foundation/FinRL)：金融 RL
- [WorldQuant Brain](https://platform.worldquantbrain.com)：免费在线因子研究平台

**视频：**
- Marcos López de Prado YouTube 系列
- Stefan Jansen "Machine Learning for Trading" 系列（基于他的同名书）

---

**下一节：Notebook 10 · 强化学习交易** — 我们会写一个 DQN 选股 agent 和 PPO 组合管理 agent。